In [1]:
!python --version

Python 3.11.14


### Breast Cancer

#### Study ieu-a-1131

https://opengwas.io/datasets/ieu-a-1131

#### Participants

- TCGA breast cancer (moste women)

#### Rout:

- map variants → genes
- build Manhattan plot
- compare multiple UKB GWAS
- integrate with your cancer pipelines (interesting cross-disease angle)

### VCF, GWAS, pysam, VEP

#### samtools - HTSlib, BCFtools

Samtools is a suite of programs for interacting with high-throughput sequencing data. It consists of three separate repositories:

  - Samtools: Reading/writing/editing/indexing/viewing SAM/BAM/CRAM format
  - BCFtools: Reading/writing BCF2/VCF/gVCF files and calling/filtering/summarising SNP and short indel sequence variants
  - HTSlib: A C library for reading/writing high-throughput sequencing data

https://www.htslib.org/


#### VEP (Variant Effect Predictor) ensembl-vep

Ensembl VEP predicts the effect of your variants (SNPs, insertions, deletions, CNVs or structural variants) on gene transcripts and protein sequence, as well as regulatory regions. It reports reference data including gene and variant phenotype associations and population allele frequencies to facilitate variant prioritisation and interpretation.


> VEP (Variant Effect Predictor) predicts the functional effects of genomic variants.  
> Haplosaurus uses phased genotype data to predict whole-transcript haplotype sequences.  
> Variant Recoder translates between different variant encodings.  


https://www.ensembl.org/info/docs/tools/vep/index.html

https://github.com/Ensembl/ensembl-vep


#### pysam

Pysam is a python module that makes it easy to read and manipulate mapped short read sequence data stored in SAM/BAM files. It is a lightweight wrapper of the htslib C-API.

https://pysam.readthedocs.io/en/latest/api.html


In [2]:
from __future__ import annotations

import os, sys
from typing import Dict, Iterable, Optional, Tuple, List
# import math
import pandas as pd
import pysam
import subprocess
import matplotlib.pyplot as plt

from pathlib import Path

ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.stat_lib import *
from libs.Basic import *



ROOT: /home/flavio/uv/VCF-GWAS
SRC added: /home/flavio/uv/VCF-GWAS/src


### 🧠 Workflow

harmonized → VEP → gene mapping

### raw x harmonized

- compare raw vs processed (quantitatively)
- detect strand flips / allele issues
- merge multiple UKB GWAS datasets
- build a robust gene-level Parkinson signature

### 👉 Harmonized = all variants are aligned to a consistent reference

That includes:

- same genome build (GRCh37 / GRCh38)
- same DNA strand (forward strand)
- consistent effect allele

In [3]:
root0 = Path("/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF")
root_vcf = root0 / 'breast_cancer/ieu-a-1131'
files = os.listdir(root_vcf)
files

['README.md', 'ieu-a-1131.vcf.gz.tbi', 'ieu-a-1131.vcf.gz']

In [4]:
min_call_rate=0.95
min_maf=0.01
require_pass=False

fname = 'ieu-a-1131.vcf.gz'
print(">>>", fname)

if fname.endswith('.vcf.gz'):
    print("Ok")
else:
    print("Wrong name, must be a vcf.gz")

filename = str(root_vcf / fname)
filename

>>> ieu-a-1131.vcf.gz
Ok


'/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF/breast_cancer/ieu-a-1131/ieu-a-1131.vcf.gz'

### Convert a VCF/BCF into:

- df         genotype matrix (samples x variants)
- y         phenotype vector (samples)
- df_ctrl    control-only matrix
- df_case    case-only matrix

#### phenotype_map:

- dict like {"sample1": 0, "sample2": 1, ...}
- where 0=normal/control and 1=disease/case

#### Filters:
- keeps only biallelic SNPs
- optional FILTER=PASS
- min call rate
- min minor allele frequency (MAF)

Missing genotypes are imputed with the variant mean dosage.

### SNPs

- SNP stands for single nucleotide polymorphism
- It is the most common type of genetic variation in people
- SNP is whe a single base pair is replaced by another
- For example, the cytosine-guanine ir replaced with a thymine-adenine

![snp](../figures/snp.png)

### ambiguous SNP

- An ambiguous SNP is one where you cannot tell strand orientation.

SNP | Problem  
A/T	looks the same after strand flip  
C/G	looks the same after strand flip  


![dna_flip](../figures/dna_flip.png)


### 🧠 Why ambiguous SNPs are dangerous

They can cause:

- flipped effects
- false associations
- incorrect meta-analysis

### 🔥 How pipelines handle them

#### Option 1 — remove them (most common)

- drop A/T and C/G SNPs

#### Option 2 — keep only if safe

If allele frequency (AF) is available:

- AF ≈ 0.5 → ambiguous → drop
- AF far from 0.5 → can infer strand


#### “Ambiguous SNPs are only ambiguous when sequence context is missing.”





### 🧬 Harmonization

> ✔ consistent allele orientation  
> ✔ same reference genome  
> ✔ comparable across datasets  


#### Ambiguous SNPs

> ❌ A/T or C/G  
> ❌ strand cannot be determined  
> ❌ often removed  


#### 🔥 In UKB files

> ukb-b-16943.vcf.gz → harmonized  
> *_raw.vcf.gz → may contain ambiguous SNPs  


### Available tools:

> detect ambiguous SNPs in your VCF  
> filter them with pysam or bcftools  
> compare raw vs harmonized impact  


### phenotype_map

In [5]:
vcf = pysam.VariantFile(filename)
vcf

### Samples

In [6]:
samples = list(vcf.header.samples)
len(samples), samples

(1, ['ieu-a-1131'])

### Format fields

In [7]:
for k in vcf.header.formats.keys():
    fmt = vcf.header.formats[k]
    print(k, fmt)

AF <pysam.libcbcf.VariantMetadata object at 0x7396457ab940>
ES <pysam.libcbcf.VariantMetadata object at 0x7396457a9a50>
SE <pysam.libcbcf.VariantMetadata object at 0x7396457aa050>
LP <pysam.libcbcf.VariantMetadata object at 0x7396457ab940>
SS <pysam.libcbcf.VariantMetadata object at 0x7396457a9a50>
EZ <pysam.libcbcf.VariantMetadata object at 0x7396457aa050>
SI <pysam.libcbcf.VariantMetadata object at 0x7396457ab940>
NC <pysam.libcbcf.VariantMetadata object at 0x7396457a9a50>
ID <pysam.libcbcf.VariantMetadata object at 0x7396457aa050>


### scan record and its samples

In [8]:
icount=0

for rec in vcf.fetch():
    icount += 1

    print(f"{icount}) id={rec.id}, chrom={rec.chrom}, pos={rec.pos}, ref={rec.ref}, alts={rec.alts}")

    print("FORMAT fields in this record:", list(rec.format.keys()))

    for sample_name, sample_data in rec.samples.items():
        print(f">>> {sample_name}")

        for key, value in sample_data.items():
            print(f"    {key}: {value}")

    if icount == 5:
        break

1) id=rs199856693, chrom=1, pos=14933, ref=G, alts=('A',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'SS', 'ID']
>>> ieu-a-1131
    ES: (-0.010400000028312206,)
    SE: (0.09700000286102295,)
    LP: (0.03886380046606064,)
    AF: (0.03889999911189079,)
    SS: (32498.0,)
    ID: rs199856693
2) id=rs374029747, chrom=1, pos=15774, ref=G, alts=('A',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'SS', 'ID']
>>> ieu-a-1131
    ES: (-0.18930000066757202,)
    SE: (0.6514000296592712,)
    LP: (0.11277700215578079,)
    AF: (0.011099999770522118,)
    SS: (32498.0,)
    ID: rs374029747
3) id=rs199745162, chrom=1, pos=16949, ref=A, alts=('C',)
FORMAT fields in this record: ['ES', 'SE', 'LP', 'AF', 'SS', 'ID']
>>> ieu-a-1131
    ES: (0.3465999960899353,)
    SE: (0.163100004196167,)
    LP: (1.4745700359344482,)
    AF: (0.013500000350177288,)
    SS: (32498.0,)
    ID: rs199745162
4) id=rs116400033, chrom=1, pos=51479, ref=T, alts=('A',)
FORMAT fields in this record: ['E

### Type of VCF

#### If one sees: ['GT', 'DP', 'AD'] --> it is genotype VCF

#### If one sees: ['ES', 'SE', 'LP', 'AF'] --> it is a summary statistics

In [10]:
formats = sorted(list(vcf.header.formats.keys()))

print("FORMAT fields:", formats)

FORMAT fields: ['AF', 'ES', 'EZ', 'ID', 'LP', 'NC', 'SE', 'SI', 'SS']


### VCF diagostic

In [11]:
print("N samples:", len(samples))
print("First samples:", samples[:5])
print("FORMAT fields:", list(vcf.header.formats.keys()))

N samples: 1
First samples: ['ieu-a-1131']
FORMAT fields: ['AF', 'ES', 'SE', 'LP', 'SS', 'EZ', 'SI', 'NC', 'ID']


### 🧬 GWAS summary fields:


| Field | 	Meaning |
|-------|-----------|
| ES | Effect size (β) |
| SE | Standard error |
| LP | −log10(p-value) |
| AF | Allele frequency |
| SS | Sample size |
| EZ | Z-score |
| SI | Study info / internal flag |
| NC | Number of cases (or count-related metric) |
| ID | Variant ID (rsID) |


Obs: rs - reference SNP ID; it is an identifier assigned by the dbSNP database.

### 📦 What information is behind an rsID

Each rsID has:

- chromosome + position
- reference allele / alternative allele
- allele frequencies (across populations)
- links to studies (GWAS, ClinVar, etc.)

### Run VEP on your VCF

```Bash
vep \
  -i UKB-b-16943.vcf.gz \
  -o UKB-b-16943_VEP.vcf.gz \
  --cache \
  --assembly GRCh38 \
  --vcf
  --fork 8
```


### Better command (for GWAS)

```Bash
vep \
  -i UKB-b-16943.vcf.gz \
  -o UKB-b-16943_VEP.vcf.gz \
  --cache \
  --assembly GRCh38 \
  --vcf \
  --symbol \
  --nearest symbol \
  --distance 10000 \
  --everything
  --fork 8
```

In [12]:
fname

'ieu-a-1131.vcf.gz'

### vep command

- But do not combine that with --no-capture-output
- If you want to inspect error text in Python, remove --no-capture-output and add capture_output=True.
- --everything is very broad and can slow runs substantially because it enables many annotations. 
- It is valid, but for a first pass you may prefer a smaller command and add options later. 
- The VEP docs describe --everything as an umbrella convenience option rather than the lightest setup

#### Do not use && in subprocess

- use "--" as argument separator; split flags between tools


In [13]:
files = os.listdir(root_vcf)
# files


fname = files[1]

input_vcf = fname
output_vcf = fname.replace('.vcf.gz', '_VEP.vcf.gz')

input_vcf, output_vcf

('ieu-a-1131.vcf.gz.tbi', 'ieu-a-1131_VEP.vcf.gz.tbi')

In [ ]:
vep_cmd = [
    "vep",
    "-i", str(input_vcf),
    "-o", str(output_vcf),
    "--cache",
    "--assembly", "GRCh38",
    "--vcf",
    "--compress_output","bgzip",
    "--symbol",
    "--nearest", "symbol",
    "--distance", "10000",
    "--everything",
    "--fork", "8",
]

vep_cmd_test = [
    "vep",
    "-i", str(input_vcf),
    "-o", str(output_vcf),
    "--cache",
    "--assembly", "GRCh38",
    "--vcf",
    "--compress_output", "bgzip",
    "--symbol",
]

tabix_command = [
    "tabix",
    "-p",
    "vcf",
    str(output_vcf)
]

# env_name = "vep"

conda_cmd = full_cmd = [
    "conda", "run",
    "-n", "vep", "--"
]
# -- argument separator; split flags between tools
#     "--no-capture-output",


In [ ]:
root_vcf

In [ ]:
full_cmd = conda_cmd + vep_cmd
full_cmd

### Run

In [ ]:
run_vep = False

if run_vep:
    try:
        result = subprocess.run(
            full_cmd,
            cwd=str(root_vcf),
            check=True,
            text=True,
            capture_output=True,
        )
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Command failed with return code {e.returncode}")
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
        raise
else:
    print("Don't run")

In [ ]:
output_vcf = 'ukb-b-16943_VEP.vcf.gz'

### 🔍 Checking ambiguous SNPs in harmonized files

In [ ]:
output_vcf2 = 'ukb-b-16943_VEP2.vcf.gz'

filename = str(root_vcf / output_vcf2)

print(">>> vep processed:", filename)

vcf = pysam.VariantFile(filename)

vcf

### Ambiguous result

> 201,994 / 1,325,664 = 15.24% means about 1 in 6.6 variants in your annotated file are A/T or C/G biallelic SNPs.  

A few important points:

1. This is not surprising.
  - Ambiguous SNPs are common enough that a value around this scale is believable.
2. This does not mean the file is bad.
  - It only means those variants are strand-ambiguous by allele label alone. In a harmonized dataset, many of them may already be safe enough for single-study downstream analysis.
3. Your count is about allele pattern, not error count.
  - You detected:

> A/T  
> T/A  
> C/G  
> G/C  

That is a structural property of the REF/ALT pair. It does not prove those variants are wrong.

In [ ]:
def is_ambiguous(ref, alt):
    pair = {ref, alt}
    return pair == {"A", "T"} or pair == {"C", "G"}

amb_count = 0
total = 0

for rec in vcf:
    if rec.alts and len(rec.alts) == 1:
        total += 1
        if is_ambiguous(rec.ref, rec.alts[0]):
            amb_count += 1

print(f"Ambiguous SNPs: {amb_count}/{total} ({amb_count/total:.2%})")

#### Extract CSQ fields

- VEP writes gene info in the CSQ field inside INFO.

In [ ]:
csq_header = vcf.header.info["CSQ"].description
csq_header

In [ ]:
csq_fields = csq_header.split("Format: ")[1].split("|")
";".join(csq_fields)

### 🔥 Dealing better with up/down mapped genes (assigned to genes)

You have:

- VEP annotations with --distance 10000
- A gene table (dfgene)
- Evidence that variants are being assigned to genes far away

#### 👉 This is expected behavior, not a bug.

- 🧠 What you should do now (clean solution)
- Instead of changing only --distance, fix it at the annotation level.


#### 👉 Add filters

- keeping only Feature_type == "Transcript"
- excluding weak proximity-only consequences:
  - upstream_gene_variant
  - downstream_gene_variant
  - intergenic_variant

### Remove weak / indirect assignments
```Python
bad = {
    "upstream_gene_variant",
    "downstream_gene_variant",
    "intergenic_variant"
}
```

- If you keep them, you get a broader transcript-level mapping.
- If you remove them, you get a more coding-focused table.


In [ ]:
def classify_consequence(consequence: str) -> str:
    terms = set(consequence.split("&"))

    if "intergenic_variant" in terms:
        return "intergenic"
    if "upstream_gene_variant" in terms or "downstream_gene_variant" in terms:
        return "near_gene"
    if "intron_variant" in terms:
        return "intronic"
    if any(x in terms for x in [
        "missense_variant",
        "synonymous_variant",
        "stop_gained",
        "stop_lost",
        "start_lost",
        "splice_donor_variant",
        "splice_acceptor_variant",
        "splice_region_variant",
        "5_prime_UTR_variant",
        "3_prime_UTR_variant"
    ]):
        return "genic"
    return "other"

In [ ]:
rows = []

# Remove weak / indirect assignments
bad = {
    "upstream_gene_variant",
    "downstream_gene_variant",
    "intergenic_variant"
}

for rec in vcf.fetch():
    sample = list(rec.samples.values())[0]

    es = sample.get("ES")
    lp = sample.get("LP")
    af = sample.get("AF")

    es = es[0] if isinstance(es, tuple) and len(es) == 1 else es
    lp = lp[0] if isinstance(lp, tuple) and len(lp) == 1 else lp
    af = af[0] if isinstance(af, tuple) and len(af) == 1 else af

    try:
        lp = float(lp)
    except:
        lp = None

    pval = 10**(-lp) if lp is not None else None

    ambiguous = (
        rec.alts is not None and
        len(rec.alts) == 1 and
        is_ambiguous(rec.ref, rec.alts[0])
    )

    csq_list = rec.info.get("CSQ")

    for csq in rec.info.get("CSQ", []):
        data = dict(zip(csq_fields, csq.split("|")))

        # variants → transcript-linked annotations
        # get value → clean it → if empty → treat as missing
        symbol  = (data.get("SYMBOL") or "").strip() or None
        ensembl = (data.get("Gene") or "").strip() or None
        gene = symbol if symbol else ensembl  # keep both separately if possible


        feature_type = data.get("Feature_type")
        consequence = data.get("Consequence", "")
        distance = data.get("DISTANCE")
        nearest = data.get("NEAREST")

        # Keep only transcript-based annotations
        # if feature_type != "Transcript":
        #     continue

        if gene:
            rows.append({
                "chrom": rec.chrom,
                "pos": rec.pos,
                "gene": gene,
                "symbol": symbol,
                "ensembl": ensembl,
                "has_symbol": symbol is not None,
                "has_ensembl": ensembl is not None,
                "variant": rec.id,
                "beta": es,
                "pval": pval,
                "af": af,
                "ambiguous": ambiguous,
                "consequence": consequence,
                "feature_type": feature_type,
                "distance": distance,
                "nearest": nearest,
            })


dfamb = pd.DataFrame(rows)
dfamb["has_gene"] = dfamb["gene"].notna()
dfamb["class"] = dfamb["consequence"].apply(classify_consequence)


print(len(dfamb))
dfamb.head(3)

In [ ]:
has_symbols = dfamb["has_symbol"].value_counts().to_list()[0]
has_ensembls = dfamb["has_ensembl"].value_counts().to_list()[0]

f"Have symbols {has_symbols/1000000:.2f} MI ensembls {has_ensembls/1000000:.2f} MI"

### New fields,no filter

> you are keeping all CSQ rows that have a gene-like field  
> you are not filtering upstream/downstream/intergenic  
> you also disabled the Feature_type == "Transcript" filter  

So dfamb is now an extended annotation table, not a strict gene-overlap table.

That’s why your first rows are:

> consequence = upstream_gene_variant
> class = near_gene

and the mapped gene is not necessarily the nearest symbol.

#### A subtle point in your example:

gene = OR4G4P or ENSG...
nearest = OR4F5

This means:

> VEP attached the consequence to a specific transcript/gene annotation row  
> the NEAREST field separately reports the nearest gene symbol  
> those do not have to be identical  

In [ ]:
dfamb.head(12)

In [ ]:
df2 = dfamb[ (dfamb["feature_type"] != "Transcript") ]
print(len(df2))
df2.head(12)


In [ ]:
dfamb_strict = dfamb[
    dfamb['has_symbol'] &
    (dfamb["feature_type"] == "Transcript") &
    ~dfamb["consequence"].apply(
        lambda s: any(term in str(s).split("&") for term in bad)
    )
].copy()

dfamb_strict = dfamb_strict.reset_index(drop=True)
print(len(dfamb_strict))
dfamb_strict.head(12)

### Strict Gene Statistics - After (dfstrict_stat)

You now keep only:

- named genes ✔
- transcript-based ✔
- not proximity-only ✔

👉 biologically meaningful gene list

In [ ]:
dfamb_strict2 = dfamb_strict.drop_duplicates(
    subset=["chrom", "gene", "variant"]
).copy()

dfstrict_stat = (
    dfamb_strict2.groupby(["chrom", "gene"])
    .agg(
        min_variant_pos=("pos", "min"),
        max_variant_pos=("pos", "max"),
        n_variants=("variant", "nunique"),
        best_pval=("pval", "min"),
        mean_beta=("beta", "mean"),
        median_beta=("beta", "median"),
        mean_af=("af", "mean"),
        n_ambiguous=("ambiguous", "sum"),
    )
    .sort_values("best_pval", ascending=True)
)

print(len(dfstrict_stat))
dfstrict_stat.head(20)

### All transcripts 🔹 Before (dfamb_stat)

- genes assigned via:
- upstream/downstream
- intergenic proximity
- weak transcript links

👉 inflated gene list

In [ ]:
dfamb2 = dfamb.drop_duplicates(subset=["chrom", "gene", "variant"]).copy()

dfamb_stat = (
    dfamb2.groupby(["chrom", "gene"])
    .agg(
        min_variant_pos=("pos", "min"),
        max_variant_pos=("pos", "max"),
        n_variants=("variant", "nunique"),
        best_pval=("pval", "min"),
        mean_beta=("beta", "mean"),
        median_beta=("beta", "median"),
        mean_af=("af", "mean"),
        n_ambiguous=("ambiguous", "sum"),
    )
    .sort_values("best_pval", ascending=True)
)

print(len(dfamb_stat))
dfamb_stat.head(20)

### Gene-linked table

In [ ]:
dfamb["has_gene"] = dfamb["gene"].notna()

dfclass = dfamb.groupby("class").size()
dfclass

### 📊 Distribution (very informative)


#### Full table

- genic      ~243k
- intronic   ~9.49M
- near_gene  ~3.94M   👈 removed later
- other      ~115k 

#### successfully removed near_gene (strict)

- genic      ~242k   ✔ almost unchanged
- intronic   ~8.22M  ⬇ reduced
- other      ~76k    ⬇ reduced
- near_gene  ❌ gone

In [ ]:
dfclass_all = dfamb.groupby(["class", "has_gene"]).size()
dfclass_all

In [ ]:
dfclass_strict = dfamb_strict.groupby(["class", "has_gene"]).size()
dfclass_strict

### Annotations - each variant can map to many transcript/gene annotations.

- about 13.8 million annotation rows
- not 13.8 million unique variants

Because **each variant can generate multiple CSQ annotations**:

- multiple transcripts
- multiple genes nearby
-  multiple consequences
- repeated Ensembl IDs / transcript-specific entries

That is why the same variant appears more than once.

In [ ]:
print(len(dfamb)/1000000, 'millions')

### 🔬 What you removed


1.09M → 0.725M

👉 ~33% of variants removed

These were mostly:

- intergenic / near-gene variants
- weak transcript assignments
- variants without reliable gene symbol

In [ ]:
variants = np.unique(dfamb_strict.variant)
nvar = len(variants)
f"There are {nvar/1000000:.3f} MI variants" 

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(dfamb_strict["pval"].dropna(), bins=50)
plt.xlabel("p-values")
plt.ylabel("Count")
plt.title("dfamb strict - pval distribution")
plt.tight_layout()
plt.show()

### Summary 🧠 Interpretation

✔ Ambiguous SNPs behave like background noise


They are:

- not enriched in significant hits
- not concentrated in key genes
- not driving your results
- ✔ Your strict gene mapping is robust

Even after:

- removing near-gene variants
- keeping only transcript-linked
- keeping only SYMBOL genes

👉 results are stable

🔥 Strong conclusion

Ambiguous SNP filtering is optional for your analysis

In [ ]:
df_noamb = dfamb_strict[~dfamb_strict["ambiguous"]].copy()

pval_threshold = 0.005

df_summ = pd.DataFrame({
    "all_strict": {
        "n_rows": len(dfamb_strict),
        "n_variants": dfamb_strict["variant"].nunique(),
        "n_genes": dfamb_strict["gene"].nunique(),
        "n_sig_rows": (dfamb_strict["pval"] < pval_threshold).sum(),
        "n_ambiguous_rows": dfamb_strict["ambiguous"].sum(),
    },
    "no_ambiguous": {
        "n_rows": len(df_noamb),
        "n_variants": df_noamb["variant"].nunique(),
        "n_genes": df_noamb["gene"].nunique(),
        "n_sig_rows": (df_noamb["pval"] < pval_threshold).sum(),
        "n_ambiguous_rows": df_noamb["ambiguous"].sum(),
    }
}).T

df_summ

### 🧬 What is beta (ES)?

In your file, ES = beta = effect size

👉 It tells you: how much the allele changes the trait


#### 🔬 In a case-control GWAS (like yours)

Beta usually comes from logistic regression:

> positive beta → increases disease risk  
> negative beta → decreases disease risk  

| Beta | Meaning |
|-----|----------|
| +	|  risk-increasing |
| −	| protective |

### 🚀 Next steps

- quantify how many SNPs were removed by harmonization
- compare gene hits raw vs clean
- detect strand issues automatically

In [ ]:
import pandas as pd

rows = []

for rec in vcf.fetch():
    for sample_name, sample_data in rec.samples.items():
        es = sample_data.get("ES")
        se = sample_data.get("SE")
        lp = sample_data.get("LP")
        af = sample_data.get("AF")
        var_id = sample_data.get("ID")

        es = es[0] if isinstance(es, tuple) and len(es) == 1 else es
        se = se[0] if isinstance(se, tuple) and len(se) == 1 else se
        lp = lp[0] if isinstance(lp, tuple) and len(lp) == 1 else lp
        af = af[0] if isinstance(af, tuple) and len(af) == 1 else af

        pval = 10 ** (-lp) if lp is not None else None

        rows.append({
            "sample": sample_name,
            "chrom": rec.chrom,
            "pos": rec.pos,
            "ref": rec.ref,
            "alt": rec.alts[0] if rec.alts else None,
            "record_id": rec.id,
            "variant_id": var_id,
            "ES": es,
            "SE": se,
            "LP": lp,
            "pval": pval,
            "AF": af,
        })

dfvar = pd.DataFrame(rows)
dfvar['fdr'] = fdr(dfvar.pval)

print(len(dfvar))

dfvar.head(3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(dfvar["fdr"].dropna(), bins=50)
plt.xlabel("FDR")
plt.ylabel("Count")
plt.title("FDR distribution")
plt.tight_layout()
plt.show()

In [ ]:
dfcool = dfvar[dfvar.fdr < 0.25]
print(len(dfcool))

dfcool

In [ ]:
def plot_manhattan(df, chrom_col="chrom", pos_col="pos", p_col="pval", figsize=(14, 6)):
    d = df[[chrom_col, pos_col, p_col]].copy()
    d = d.dropna()
    d = d[d[p_col] > 0].copy()

    # keep autosomes/X/Y if needed; here just sort naturally if possible
    def chrom_key(x):
        x = str(x).replace("chr", "")
        if x == "X":
            return 23
        if x == "Y":
            return 24
        try:
            return int(x)
        except ValueError:
            return 10**9

    d["chrom_sort"] = d[chrom_col].map(chrom_key)
    d = d.sort_values(["chrom_sort", pos_col]).copy()

    d["mlog10p"] = -np.log10(d[p_col])

    # cumulative genomic position
    xvals = []
    xticks = []
    xlabels = []
    offset = 0

    for chrom, sub in d.groupby(chrom_col, sort=False):
        sub = sub.sort_values(pos_col)
        x = sub[pos_col].values + offset
        xvals.extend(x)

        xticks.append(x.mean())
        xlabels.append(str(chrom))

        offset = x.max()

    d["x"] = xvals

    plt.figure(figsize=figsize)

    for i, (chrom, sub) in enumerate(d.groupby(chrom_col, sort=False)):
        plt.scatter(
            sub["x"],
            sub["mlog10p"],
            s=8,0.
            alpha=0.7,
            label=str(chrom) if i < 5 else None
        )

    # genome-wide significance line
    plt.axhline(-np.log10(5e-8), linestyle="--", linewidth=1)

    plt.xticks(xticks, xlabels, rotation=0)
    plt.xlabel("Chromosome")
    plt.ylabel("-log10(p-value)")
    plt.title("Manhattan plot")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_manhattan(dfvar, chrom_col="chrom", pos_col="pos", p_col="pval", figsize=(14, 6))

In [ ]:
sorted(dfvar["chrom"].unique())

In [ ]:
def plot_manhattan_highlight(df, chrom_col="chrom", pos_col="pos", p_col="pval", fdr_col="fdr", fdr_thr=0.25, figsize=(14,6)):
    d = df[[chrom_col, pos_col, p_col, fdr_col]].copy()
    d = d.dropna()
    d = d[d[p_col] > 0].copy()

    def chrom_key(x):
        x = str(x).replace("chr", "")
        if x == "X":
            return 23
        if x == "Y":
            return 24
        try:
            return int(x)
        except ValueError:
            return 10**9

    d["chrom_sort"] = d[chrom_col].map(chrom_key)
    d = d.sort_values(["chrom_sort", pos_col]).copy()
    d["mlog10p"] = -np.log10(d[p_col])

    xvals = []
    xticks = []
    xlabels = []
    offset = 0

    for chrom, sub in d.groupby(chrom_col, sort=False):
        sub = sub.sort_values(pos_col)
        x = sub[pos_col].values + offset
        xvals.extend(x)
        xticks.append(x.mean())
        xlabels.append(str(chrom))
        offset = x.max()

    d["x"] = xvals

    base = d[d[fdr_col] >= fdr_thr]
    hit = d[d[fdr_col] < fdr_thr]

    plt.figure(figsize=figsize)
    plt.scatter(base["x"], base["mlog10p"], s=8, alpha=0.5)
    plt.scatter(hit["x"], hit["mlog10p"], s=10, alpha=0.9)

    plt.axhline(-np.log10(5e-8), linestyle="--", linewidth=1)
    plt.xticks(xticks, xlabels)
    plt.xlabel("Chromosome")
    plt.ylabel("-log10(p-value)")
    plt.title(f"Manhattan plot (highlight FDR < {fdr_thr})")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_manhattan_highlight(dfvar, fdr_thr=0.25)

### 🔬 Results: Top hits:

All strongest variants are in:

> chr5 : ~13.1 – 13.3 Mb  
> locus = 13  

👉 That is a single genomic locus (cluster)

#### 🧠 Interpretation

> 🔥 This is a real GWAS signal  

**Why?**

- ✔ many SNPs
- ✔ very close positions
- ✔ similar p-values
- ✔ similar effect sizes

👉 This is **linkage disequilibrium (LD)**

In [ ]:
dfsig = dfvar[dfvar["pval"] < 1e-5].copy()
dfsig["locus"] = dfsig["pos"] // 1_000_000
dfg = dfsig.groupby(["chrom", "locus"]).size()

dfg

In [ ]:
dfsig = dfsig.sort_values("fdr", ascending=True)
dfsig

In [ ]:
dfamb.head(2)

In [ ]:
idx = dfamb2.groupby(["chrom", "gene"])["pval"].idxmin()

best_variants = (
    dfamb2.loc[idx, ["chrom", "gene", "variant", "pval", "beta", "af"]]
    .rename(columns={
        "variant": "best_variant",
        "pval": "best_variant_pval",
        "beta": "best_variant_beta",
        "af": "best_variant_af"
    })
)

dfgene_reset = dfgene.reset_index()
dfgene_final = dfgene_reset.merge(best_variants, on=["chrom", "gene"], how="left")
dfgene_final.head(20)

🚀 Next steps

- fetch info for rsIDs automatically
- link rsIDs → genes → pathways
- enrich your Parkinson results with known biology